# Notebook 7 . Ubah Bentuk Tabel

Ubah tabel lebar menjadi panjang (wide ke long) atau panjang menjadi lebar (long ke wide). Contohnya, kolom penjualan per bulan dapat diubah menjadi satu kolom bulan dan satu kolom nilai.

In [ ]:
from pathlib import Path
import pandas as pd

# PENGATURAN YANG BISA DIUBAH
Input_CSV = Path("Input/input.csv")
Separator_CSV = ","  # Ganti menjadi ";" jika CSV memakai titik koma
Cara_Ubah = "lebar_ke_panjang"  # atau "panjang_ke_lebar"
# Pengaturan untuk ubah tabel lebar menjadi panjang:
Kolom_ID = ["id"]
Kolom_Nilai_Sumber = ["jan", "feb", "mar"]
Nama_Kolom_Variabel = "bulan"
Nama_Kolom_Nilai = "nilai"
# Pengaturan untuk ubah tabel panjang menjadi lebar:
Kolom_Indeks = ["id"]
Kolom_Pivot = "bulan"
Nilai_Pivot = "nilai"
Cara_Gabung_Nilai = "pertama"  # "pertama", "jumlah", "rata_rata", "minimum", "maksimum"
Output_CSV = Path("Output/hasil_reshape.csv")

def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Ubah bentuk dan simpan

In [ ]:
source = Input_CSV if Input_CSV.is_absolute() else BASE_DIR / Input_CSV
if not source.exists(): raise FileNotFoundError(f"File tidak ditemukan: {source}")
df = pd.read_csv(source, sep=Separator_CSV, dtype="string", encoding="utf-8-sig")
if Cara_Ubah == "lebar_ke_panjang":
    needed = Kolom_ID + Kolom_Nilai_Sumber
    missing = [c for c in needed if c not in df]
    if missing: raise KeyError(f"Kolom tidak ditemukan: {missing}")
    result = df.melt(id_vars=Kolom_ID, value_vars=Kolom_Nilai_Sumber, var_name=Nama_Kolom_Variabel, value_name=Nama_Kolom_Nilai)
elif Cara_Ubah == "panjang_ke_lebar":
    needed = Kolom_Indeks + [Kolom_Pivot, Nilai_Pivot]
    missing = [c for c in needed if c not in df]
    if missing: raise KeyError(f"Kolom tidak ditemukan: {missing}")
    pemetaan_agregasi = {"pertama": "first", "jumlah": "sum", "rata_rata": "mean", "minimum": "min", "maksimum": "max"}
    if Cara_Gabung_Nilai not in pemetaan_agregasi:
        raise ValueError("Cara_Gabung_Nilai harus: pertama, jumlah, rata_rata, minimum, atau maksimum.")
    if Cara_Gabung_Nilai in {"jumlah", "rata_rata"}: df[Nilai_Pivot] = pd.to_numeric(df[Nilai_Pivot], errors="coerce")
    result = df.pivot_table(index=Kolom_Indeks, columns=Kolom_Pivot, values=Nilai_Pivot, aggfunc=pemetaan_agregasi[Cara_Gabung_Nilai]).reset_index()
    result.columns.name = None
else:
    raise ValueError("Cara_Ubah harus 'lebar_ke_panjang' atau 'panjang_ke_lebar'.")

display(result.head())
output = Output_CSV if Output_CSV.is_absolute() else BASE_DIR / Output_CSV
output.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output, index=False, encoding="utf-8-sig", sep=Separator_CSV)
print(f"Tersimpan: {output.resolve()} | {len(result):,} baris x {len(result.columns)} kolom")
